In [ ]:
import os
import json
import re

import gradio as gr

from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_ollama import ChatOllama

In [ ]:
documents = []
vectorstore = None
retriever = None

flashcards = []
current_card = 0

In [ ]:
llm = ChatOllama(
    model="qwen3:8b",
    temperature=0,
    think=False
)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
def process_pdf(pdf_file):

    reader = PdfReader(pdf_file)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_text(text)

    docs = []

    for i, chunk in enumerate(chunks):

        docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "chunk_id": i
                }
            )
        )

    return docs

In [ ]:
def upload_pdf(pdf_file):

    global documents
    global vectorstore
    global retriever

    if pdf_file is None:
        return "Please upload a PDF."

    documents = process_pdf(pdf_file)

    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 5}
    )

    return (
        f"PDF processed successfully.\n"
        f"Chunks: {len(documents)}"
    )

In [ ]:
def chat(message, history):

    if retriever is None:

        return (
            "Please upload a PDF first."
        )

    docs = retriever.invoke(message)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    history_text = ""

    if history:

        for item in history[-6:]:

            if item["role"] == "user":

                history_text += (
                    f"User: {item['content']}\n"
                )

            elif item["role"] == "assistant":

                history_text += (
                    f"Assistant: {item['content']}\n"
                )

    prompt = f"""
You are a PDF assistant.

If the user says:
- Hi
- Hello
- Hey

then greet them and ask how you can help with the PDF.

Previous Conversation:

{history_text}

Context:

{context}

Current Question:

{message}

Rules:

1. Answer ONLY using the provided context.
2. Use previous conversation when resolving references such as:
   - it
   - they
   - this
   - that
   - those
3. Do not use outside knowledge.
4. If the answer is not found in the context, say:

"I cannot find this information in the uploaded PDF."

5. Be concise.
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
def generate_summary():

    if not documents:
        return "Upload a PDF first."

    text = "\n\n".join(
        doc.page_content
        for doc in documents[:30]
    )

    prompt = f"""
Generate a concise summary of the document.

Content:

{text}

Include:

1. Problem Statement
2. Proposed Solution
3. Methodology
4. Results
5. Conclusion

Maximum 500 words.
Do not inlcude the word count unless specifically asked.
"""

    response= llm.invoke(prompt)
    summary = response.content

    with open(
        "summary.txt",
        "w",
        encoding="utf-8"
    ) as f:

        f.write(summary)

    return (
        summary,
        "summary.txt"
    )

In [ ]:
def generate_quiz(difficulty):

    if not documents:
        return "Upload a PDF first."

    text = "\n\n".join(
        doc.page_content
        for doc in documents[:15]
    )

    difficulty_rules = {

    "Easy":
    "Generate simple factual and definition-based questions.",

    "Medium":
    "Generate concept-based and application-oriented questions.",

    "Hard":
    "Generate analytical and reasoning-based questions.",

    "Mixed":
    "Generate a mix of easy, medium, and hard questions."
}

    prompt = f"""
Generate EXACTLY 10 Multiple Choice Questions.

Difficulty:
{difficulty}

Instructions:
{difficulty_rules[difficulty]}

Rules:

- Use ONLY information from the document.
- Each question must have a complete question statement.
- Each question must have exactly 4 options.
- Include the correct answer.
- Include a short explanation.
- Do not generate summaries.
- Do not generate notes.
- Randomize the position of the correct answer.
- Distribute correct answers across A, B, C, and D.

Format:

Question 1:
Question text

A) Option A
B) Option B
C) Option C
D) Option D

Answer: A

Explanation:
Explanation text

Question 2:
...

Content:

{text}
"""
    response = llm.invoke(prompt)

    quiz = response.content.strip()

    print("\n===== QUIZ RESPONSE =====")
    print(quiz)
    print("=========================\n")

    return quiz

In [ ]:
def generate_flashcards():

    global flashcards
    global current_card

    if not documents:
        return (
            "Upload a PDF first.",
            ""
        )

    text = "\n\n".join(
        doc.page_content
        for doc in documents[:15]
    )

    prompt = f"""
Generate EXACTLY 5 flashcards.

Return ONLY valid JSON.

Format:

[
  {{
    "front": "Question 1",
    "back": "Answer 1"
  }},
  {{
    "front": "Question 2",
    "back": "Answer 2"
  }},
  {{
    "front": "Question 3",
    "back": "Answer 3"
  }},
  {{
    "front": "Question 4",
    "back": "Answer 4"
  }},
  {{
    "front": "Question 5",
    "back": "Answer 5"
  }}
]

Rules:

- Generate exactly 5 flashcards.
- Each flashcard must cover a different concept.
- No markdown.
- No explanation.
- No notes.
- Output ONLY JSON.

Content:

{text}
"""

    response = llm.invoke(prompt)

    content = response.content.strip()

    print("\n===== FLASHCARD RESPONSE =====")
    print(content)
    print("=============================\n")

    content = re.sub(
        r"<think>.*?</think>",
        "",
        content,
        flags=re.DOTALL
    )

    content = content.replace(
        "```json",
        ""
    )

    content = content.replace(
        "```",
        ""
    ).strip()

    try:

        match = re.search(
            r"\[.*\]",
            content,
            re.DOTALL
        )

        if match:
            content = match.group()

        flashcards = json.loads(content)

        if not isinstance(
            flashcards,
            list
        ):
            raise ValueError(
                "JSON is not a list"
            )

        if len(flashcards) == 0:
            raise ValueError(
                "No flashcards returned"
            )

        current_card = 0

        card = flashcards[0]

        return (
            f"""
### Question

{card['front']}
""",
            "Answer Hidden"
        )

    except Exception as e:

        print("\n===== FLASHCARD ERROR =====")
        print(e)

        print("\n===== CONTENT RECEIVED =====")
        print(content)

        return (
            "Failed to generate flashcards.",
            ""
        )

In [ ]:
def previous_flashcard():

    global current_card

    if not flashcards:
        return "", ""

    current_card -= 1

    if current_card < 0:
        current_card = len(flashcards) - 1

    card = flashcards[current_card]

    return (
        f"### Question\n\n{card['front']}",
        "Answer Hidden"
    )

In [ ]:
def next_flashcard():

    global current_card

    if not flashcards:
        return "", ""

    current_card += 1

    if current_card >= len(flashcards):
        current_card = 0

    card = flashcards[current_card]

    return (
        f"### Question\n\n{card['front']}",
        "Answer Hidden"
    )

In [ ]:
def reveal_answer():

    if not flashcards:
        return "", ""

    card = flashcards[current_card]

    return (
        f"### Question\n\n{card['front']}",
        f"### Answer\n\n{card['back']}"
    )

In [ ]:
def reset_flashcards():

    global flashcards
    global current_card

    flashcards = []
    current_card = 0

    return (
        "Flashcards cleared.",
        ""
    )

In [ ]:
def generate_notes():

    if not documents:
        return "Upload a PDF first."

    text = "\n\n".join(
        doc.page_content
        for doc in documents[:30]
    )

    prompt = f"""
Create exam-oriented study notes from the document.

Content:

{text}

Generate the following sections:

1. Overview
2. Key Concepts
3. Important Definitions
4. Architecture / Methodology
5. Important Findings
6. Advantages
7. Limitations
8. Future Scope
9. Viva Questions 
10. Interview Questions
11. Exam Revision Points

Rules:

- Use bullet points.
- Keep notes concise.
- Focus on concepts rather than summarizing the paper.
- Do not write an abstract.
- Do not write a research paper summary.
-The number of questions in sections 9, 10 and 11 should be 5 each.
"""

    response= llm.invoke(prompt)
    notes = response.content

    with open(
        "notes.txt",
        "w",
        encoding="utf-8"
    ) as f:

        f.write(notes)

    return (
        notes,
        "notes.txt"
    )

In [ ]:
def generate_glossary():

    if not documents:
        return "Upload a PDF first."

    text = "\n\n".join(
        doc.page_content
        for doc in documents[:15]
    )

    prompt = f"""
You are generating a glossary.

TASK:
Extract important terms from the document and define them.

Rules:
- Output ONLY glossary entries.
- Do NOT summarize.
- Do NOT continue the document.
- Do NOT write conclusions.
- Do NOT write future directions.
- Do NOT write paragraphs.

Format:

Term: <term>
Definition: <definition>

Term: <term>
Definition: <definition>

Generate exactly 15 glossary entries.

Document:

{text}
"""

    response = llm.invoke(prompt)
    print("\n===== GLOSSARY RESPONSE =====")
    print(response.content)
    print("============================\n")
    return response.content

In [ ]:
import gradio as gr

with gr.Blocks(
    title="Smart Learning Companion"
) as demo:

    gr.Markdown(
        """
# Smart Learning Companion
A tool to help you learn from PDFs using LLMs.
"""
    )

    # ==========================
    # Upload
    # ==========================

    with gr.Tab("Upload"):

        pdf_input = gr.File(
            file_types=[".pdf"],
            label="Upload PDF"
        )

        upload_btn = gr.Button(
            "Process PDF"
        )

        upload_output = gr.Textbox(
            label="Status"
        )

        upload_btn.click(
            upload_pdf,
            inputs=pdf_input,
            outputs=upload_output
        )

    # ==========================
    # Chat
    # ==========================

    with gr.Tab("Chat"):

        gr.ChatInterface(
            fn=chat,
            title="Chat with PDF"
        )

    # ==========================
    # Summary
    # ==========================

    with gr.Tab("Summary"):

        summary_btn = gr.Button(
            "Generate Summary"
        )

        summary_output = gr.Markdown()

        summary_file = gr.File(
            label="Download Summary")

        summary_btn.click(
            generate_summary,
            outputs=[summary_output,summary_file]
        )

    # ==========================
    # Quiz
    # ==========================

    with gr.Tab("Quiz"):

        difficulty_quiz=gr.Dropdown(
            choices=["Easy", "Medium", "Hard", "Mixed"],
            label="Select Difficulty",
            value="Mixed"
        )
        quiz_btn = gr.Button(
            "Generate Quiz"
        )

        quiz_output = gr.Markdown()

        quiz_btn.click(
            generate_quiz,
            inputs=difficulty_quiz,
            outputs=quiz_output
        )

    # ==========================
    # Flashcards
    # ==========================

    with gr.Tab("Flashcards"):

        flash_btn = gr.Button(
            "Generate Flashcards"
        )

        front_box = gr.Markdown(
            "Generate flashcards to begin."
        )

        back_box = gr.Markdown()

        with gr.Row():

            previous_btn = gr.Button(
                "Previous"
            )

            reveal_btn = gr.Button(
                "Reveal Answer"
            )

            next_btn = gr.Button(
                "Next"
            )

            reset_btn = gr.Button(
                "Reset"
            )

        flash_btn.click(
            generate_flashcards,
            outputs=[
                front_box,
                back_box
            ]
        )

        previous_btn.click(
            previous_flashcard,
            outputs=[
                front_box,
                back_box
            ]
        )

        reveal_btn.click(
            reveal_answer,
            outputs=[
                front_box,
                back_box
            ]
        )

        next_btn.click(
            next_flashcard,
            outputs=[
                front_box,
                back_box
            ]
        )

        reset_btn.click(
            reset_flashcards,
            outputs=[
                front_box,
                back_box
            ]
        )

    # ==========================
    # Notes
    # ==========================

    with gr.Tab("Notes"):

        notes_btn = gr.Button(
            "Generate Notes"
        )

        notes_output = gr.Markdown()

        notes_file = gr.File(
            label="Download Notes"
        )
        
        notes_btn.click(
            generate_notes,
            outputs=[notes_output, notes_file]
        )

    with gr.Tab("Glossary"):

        glossary_btn = gr.Button(
        "Generate Glossary"
        )

        glossary_output = gr.Markdown()

        glossary_btn.click(
        generate_glossary,
        outputs=glossary_output
        )

demo.queue()

demo.launch(
    inbrowser=True
)